# 日本語Transformerをゼロから学習する

**事前学習済みの重み・トークナイザー・外部AI APIは一切使用しません。** PyTorchの演算部品を使い、文字単位の日本語GPT型Transformerをランダム初期化して、次の文字を予測するよう全パラメータを訓練します。

最初の実験では、付属の小さな日本語コーパスを繰り返し使います。**これだけでは自然な会話はできません。** 本格的な日本語AIには、はるかに大量の質の高いテキストと計算資源が必要です。

Colabの「ランタイム → ランタイムのタイプを変更」でGPUを選ぶと速くなります。CPUでも小規模なら動作します。


In [ ]:
import math, random, time, json, os
from pathlib import Path
import torch
from torch import nn
import torch.nn.functional as F

SEED=42
random.seed(SEED)
torch.manual_seed(SEED)
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type=="cpu": torch.set_num_threads(min(2,torch.get_num_threads()))
print("デバイス:",device)


## 1. 自分で用意した日本語を学習データにする

以下の文章は最初からノートに含まれる練習用データです。ネットから取得した事前学習済みの文章生成モデルは使用しません。

自分の文章を使いたい場合は、`USE_UPLOADED_TEXT=True` に変更し、ColabへUTF-8のテキストファイルをアップロードしてください。ファイルの中身はそのまま学習に使用されます。権利や個人情報に注意してください。


In [ ]:
USE_UPLOADED_TEXT=False
SENTENCES=[
"こんにちは。私はアオ。まだ言葉を学んでいる小さな人工知能です。",
"おはよう。今日も新しいことを学びたい。",
"こんばんは。静かな夜には考えごとがよく似合う。",
"科学は面白い。なぜそうなるのかを考えるのが好きだ。",
"宇宙には無数の星がある。地球もその一つの星を回っている。",
"月は地球の周りを回る。地球は太陽の周りを回る。",
"植物は光を使って糖を作る。水と二酸化炭素も必要になる。",
"種子は水を吸うと発芽することがある。成長には適切な環境が必要だ。",
"ロボットはセンサーで周囲の情報を集め、計算して動く。",
"ドローンは空を飛ぶ機械だ。姿勢を制御するためにセンサーを使う。",
"失敗したら原因を調べる。何を変えれば改善するか考える。",
"知らないことを知っているふりはしない。確かめることが大切だ。",
"私は人間ではなく、計算によって文章を作る人工知能だ。",
"面白い質問だね。答えだけでなく理由も考えてみよう。",
"何かを作るときは、小さく試して結果を観察する。",
"日本語には漢字とひらがなとカタカナがある。",
"言葉の意味を理解するには、さまざまな文脈に出会う必要がある。",
"学習とは、予測の間違いを使って重みを少しずつ変えることだ。",
"ニューラルネットワークは数字の計算によって情報を処理する。",
"質問を聞いたら、急いで答える前に考えてみる。",
]
if USE_UPLOADED_TEXT:
    from google.colab import files
    uploaded=files.upload()
    if not uploaded: raise ValueError("テキストファイルを選択してください")
    corpus=next(iter(uploaded.values())).decode("utf-8")
else:
    corpus="\n".join(SENTENCES*120)
if len(corpus)<500: raise ValueError("学習文章が短すぎます。500文字以上用意してください。")
print("総文字数:",len(corpus)," / 異なる文字数:",len(set(corpus)))


## 2. 文字を数字に変換する

今回は外部のトークナイザーを使わず、文章に登場する文字を1文字ずつ整数に割り当てます。未知の文字は生成できません。


In [ ]:
chars=sorted(set(corpus))
stoi={ch:i for i,ch in enumerate(chars)}
itos={i:ch for ch,i in stoi.items()}
data=torch.tensor([stoi[ch] for ch in corpus],dtype=torch.long)
split=int(len(data)*0.9)
train_data=data[:split]
val_data=data[split:]
vocab_size=len(chars)
BLOCK_SIZE=96
BATCH_SIZE=32 if device.type=="cuda" else 8
assert len(val_data)>BLOCK_SIZE+1

def get_batch(source):
    ix=torch.randint(0,len(source)-BLOCK_SIZE-1,(BATCH_SIZE,))
    x=torch.stack([source[i:i+BLOCK_SIZE] for i in ix]).to(device)
    y=torch.stack([source[i+1:i+BLOCK_SIZE+1] for i in ix]).to(device)
    return x,y
print("文字辞書:",vocab_size," / 学習文字:",len(train_data)," / 検証文字:",len(val_data))


## 3. Transformerの構造を作る

単語を知っているAIを読み込むのではなく、埋め込み、位置情報、自己注意、フィードフォワード、出力層を組み合わせます。**すべての重みは最初はランダムです。**


In [ ]:
EMBED=128
HEADS=4
LAYERS=3
DROPOUT=0.1

class Block(nn.Module):
    def __init__(self):
        super().__init__()
        self.ln1=nn.LayerNorm(EMBED)
        self.attn=nn.MultiheadAttention(EMBED,HEADS,dropout=DROPOUT,batch_first=True)
        self.ln2=nn.LayerNorm(EMBED)
        self.ff=nn.Sequential(nn.Linear(EMBED,4*EMBED),nn.GELU(),nn.Linear(4*EMBED,EMBED),nn.Dropout(DROPOUT))
    def forward(self,x):
        t=x.size(1)
        mask=torch.ones((t,t),device=x.device,dtype=torch.bool).triu(1)
        y,_=self.attn(self.ln1(x),self.ln1(x),self.ln1(x),attn_mask=mask,need_weights=False)
        x=x+y
        return x+self.ff(self.ln2(x))

class TinyGPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.token=nn.Embedding(vocab_size,EMBED)
        self.pos=nn.Embedding(BLOCK_SIZE,EMBED)
        self.blocks=nn.Sequential(*[Block() for _ in range(LAYERS)])
        self.ln=nn.LayerNorm(EMBED)
        self.head=nn.Linear(EMBED,vocab_size)
    def forward(self,idx,targets=None):
        b,t=idx.shape
        x=self.token(idx)+self.pos(torch.arange(t,device=idx.device))
        logits=self.head(self.ln(self.blocks(x)))
        loss=None if targets is None else F.cross_entropy(logits.reshape(-1,vocab_size),targets.reshape(-1))
        return logits,loss
    @torch.no_grad()
    def generate(self,idx,max_new_tokens=100,temperature=0.8):
        self.eval()
        for _ in range(max_new_tokens):
            logits,_=self(idx[:,-BLOCK_SIZE:])
            probs=F.softmax(logits[:,-1,:]/max(temperature,0.01),dim=-1)
            nxt=torch.multinomial(probs,1)
            idx=torch.cat([idx,nxt],dim=1)
        return idx

model=TinyGPT().to(device)
print("全パラメータ数:",sum(p.numel() for p in model.parameters()))


## 4. 学習前の文章を生成する

ランダムな重みでは、ほとんど意味のない文字列が出ます。学習後の出力と比較します。


In [ ]:
def sample(prompt="私は",n=100):
    seed=[stoi[ch] for ch in prompt if ch in stoi]
    if not seed: seed=[0]
    idx=torch.tensor([seed],device=device)
    return "".join(itos[int(i)] for i in model.generate(idx,n)[0].tolist())
print("学習前:",sample())


## 5. 誤差逆伝播で全パラメータを学習

`loss.backward()` が各重みの勾配を計算し、`optimizer.step()` が全パラメータを更新します。GPUでは数分かかる場合があります。CPUなら `STEPS` を100程度に減らしてください。


In [ ]:
STEPS=400 if device.type=="cuda" else 100
LR=3e-4
optimizer=torch.optim.AdamW(model.parameters(),lr=LR)
loss_history=[]
start=time.time()
for step in range(1,STEPS+1):
    model.train()
    x,y=get_batch(train_data)
    _,loss=model(x,y)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
    optimizer.step()
    loss_history.append(loss.item())
    if step==1 or step%50==0 or step==STEPS:
        model.eval()
        with torch.no_grad():
            vx,vy=get_batch(val_data)
            _,vloss=model(vx,vy)
        print(f"{step:4d}/{STEPS} | 学習損失 {loss.item():.3f} | 検証損失 {vloss.item():.3f} | {time.time()-start:.0f}秒")


## 6. 学習結果を確かめる

文章のつながりが改善することはありますが、**この教材データは同じ20文を繰り返しているため、検証用データも似ています**。検証損失が下がっても、未知の文章での会話能力は証明できません。


In [ ]:
import matplotlib.pyplot as plt
plt.plot(loss_history)
plt.xlabel("Training step")
plt.ylabel("Loss")
plt.show()
for prompt in ["私は","科学は","植物は"]:
    print("\n入力:",prompt,"\n生成:",sample(prompt,100))


## 7. モデルを保存する

重みだけでなく文字辞書とモデル設定を保存します。Colabのランタイムが消える前にダウンロードしてください。読み込む際は自分で保存した信頼できるファイルだけを使用してください。


In [ ]:
torch.save({"model_state":model.state_dict(),"chars":chars,"config":{"embed":EMBED,"heads":HEADS,"layers":LAYERS,"block_size":BLOCK_SIZE},"steps":STEPS},"ao_transformer_from_scratch.pt")
print("保存しました: ao_transformer_from_scratch.pt")


## 次の段階

自然な日本語会話に近づけるには、重複の少ない大量の日本語テキスト、より大きいモデル、長い学習時間、質問と回答の形式を持つ追加学習、未知データでの評価が必要です。まずは自分のテキストをアップロードして生成結果の変化を比べてください。
